In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Performance Management: funkcja Qiskit dostarczana przez Q-CTRL Fire Opal
*Zobacz [dokumentację API](https://docs.quantum.ibm.com/api/functions/q-ctrl-performance-management)*

> **Note:** Funkcje Qiskit to funkcja eksperymentalna dostępna wyłącznie dla użytkowników planów IBM Quantum&reg; Premium Plan, Flex Plan oraz On-Prem (za pośrednictwem IBM Quantum Platform API). Są one w fazie wstępnej wersji (preview) i mogą ulec zmianie.


<Accordion>
<AccordionItem title="Wersje pakietów">

Kod na tej stronie został opracowany z wykorzystaniem poniższych wymagań.
Zalecamy użycie tych wersji lub nowszych.

```
qiskit[all]~=2.3.1
qiskit-ibm-runtime~=0.45.1
```
</AccordionItem>
</Accordion>

## Przegląd
Fire Opal Performance Management sprawia, że każdy może uzyskiwać znaczące wyniki z komputerów kwantowych na dużą skalę, bez konieczności bycia ekspertem od sprzętu kwantowego. Podczas uruchamiania obwodów z Fire Opal Performance Management automatycznie stosowane są oparte na AI techniki tłumienia błędów, umożliwiając skalowanie większych problemów z większą liczbą bramek i qubit. Takie podejście zmniejsza liczbę shotów potrzebnych do uzyskania prawidłowej odpowiedzi, bez żadnych dodatkowych kosztów — co przekłada się na znaczne oszczędności zarówno czasu obliczeniowego, jak i kosztów.

Performance Management tłumi błędy i zwiększa prawdopodobieństwo uzyskania właściwej odpowiedzi na zaszumionym sprzęcie. Innymi słowy, zwiększa stosunek sygnału do szumu. Poniższy rysunek pokazuje, jak zwiększona dokładność dzięki Performance Management może zmniejszyć potrzebę dodatkowych shotów w przypadku algorytmu kwantowej transformaty Fouriera na 10 qubit. Przy zaledwie 30 shotach Q-CTRL osiąga próg pewności 99%, podczas gdy konfiguracja domyślna (`QiskitRuntime` Sampler, `optimization_level`=3 i `resilience_level`=1, `ibm_sherbrooke`) wymaga 170 000 shotów. Uzyskując właściwą odpowiedź szybciej, oszczędzasz znaczny czas obliczeniowy.

![Wizualizacja poprawionego czasu wykonania](../docs/images/guides/qctrl-performance-management/achieve_more.svg)

Funkcja Performance Management może być używana z dowolnym algorytmem, a jej zastosowanie zamiast standardowych [prymitywów Qiskit Runtime](/guides/primitives) jest proste. Za kulisami wiele technik tłumienia błędów współpracuje ze sobą, aby zapobiegać błędom w trakcie wykonywania. Wszystkie metody potoku Fire Opal są wstępnie skonfigurowane i niezależne od algorytmu, co oznacza, że zawsze uzyskujesz najlepszą wydajność bez żadnej konfiguracji.

Aby uzyskać dostęp do Performance Management, [skontaktuj się z Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Opis
Fire Opal Performance Management oferuje dwie opcje wykonania podobne do prymitywów Qiskit Runtime, dzięki czemu możesz łatwo zamienić je na Q-CTRL Sampler i Estimator. Ogólny przepływ pracy przy użyciu funkcji Performance Management jest następujący:
1. Zdefiniuj swój obwód (oraz operatory w przypadku Estimator).
2. Uruchom obwód.
3. Pobierz wyniki.

Aby zredukować szum sprzętowy, Fire Opal stosuje szereg opartych na AI technik tłumienia błędów przedstawionych na poniższym rysunku. Dzięki Fire Opal cały potok jest w pełni zautomatyzowany i nie wymaga żadnej konfiguracji.

Potok Fire Opal eliminuje potrzebę ponoszenia dodatkowych kosztów, takich jak zwiększony czas wykonywania na sprzęcie kwantowym czy dodatkowe fizyczne qubit. Należy pamiętać, że klasyczny czas przetwarzania nadal jest czynnikiem (więcej szacunków znajdziesz w sekcji [Benchmarki](#benchmarks), gdzie „Łączny czas" obejmuje zarówno klasyczne, jak i kwantowe przetwarzanie). W przeciwieństwie do łagodzenia błędów, które wymaga dodatkowych kosztów w postaci próbkowania, tłumienie błędów Fire Opal działa zarówno na poziomie bramki, jak i pulsu, aby adresować różne źródła szumu i zapobiegać wystąpieniu błędów. Zapobiegając błędom, eliminuje się potrzebę kosztownego przetwarzania końcowego.

Poniższy rysunek przedstawia metody tłumienia błędów zautomatyzowane przez Fire Opal Performance Management.

![Wizualizacja potoku tłumienia błędów](../docs/images/guides/qctrl-performance-management/error_suppression.svg)

Funkcja oferuje dwa prymitywy, Sampler i Estimator, a wejścia i wyjścia obu rozszerzają zaimplementowaną specyfikację dla [prymitywów Qiskit Runtime V2](/guides/primitive-input-output#pubs).
## Benchmarki
[Opublikowane wyniki benchmarkowania algorytmicznego](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.024034) wykazują znaczącą poprawę wydajności w różnych algorytmach, w tym Bernstein-Vazirani, kwantowej transformacie Fouriera, wyszukiwaniu Grovera, kwantowym algorytmie optymalizacji przybliżonej oraz variational quantum eigensolver. W dalszej części tej sekcji znajdują się szczegółowe informacje o typach algorytmów, które możesz uruchamiać, a także o oczekiwanej wydajności i czasach wykonywania.

Poniższe niezależne badania pokazują, jak Performance Management Q-CTRL umożliwia prowadzenie badań algorytmicznych na rekordową skalę:
- [Parametrized Energy-Efficient Quantum Kernels for Network Service Fault Diagnosis](https://arxiv.org/abs/2405.09724v1) – kwantowe uczenie jądrowe na do 50 qubit
- [Tensor-based quantum phase difference estimation for large-scale demonstration](https://arxiv.org/abs/2408.04946) – kwantowe szacowanie fazy na do 33 qubit
- [Hierarchical Learning for Quantum ML: Novel Training Technique for Large-Scale Variational Quantum Circuits](https://arxiv.org/abs/2311.12929) – kwantowe ładowanie danych na do 21 qubit

Poniższa tabela stanowi przybliżony przewodnik po dokładności i czasach wykonywania z poprzednich uruchomień benchmarkowych na `ibm_fez`. Wydajność na innych urządzeniach może się różnić. Czas użytkowania oparty jest na założeniu 10 000 shotów na obwód. Wskazana „Liczba qubit" nie jest twardym ograniczeniem, lecz reprezentuje przybliżone progi, przy których można oczekiwać bardzo spójnej dokładności rozwiązania. Większe problemy zostały z powodzeniem rozwiązane i zachęcamy do testowania powyżej tych limitów.

| Przykład    | Liczba qubit | Dokładność | Miara dokładności | Łączny czas (s) | Czas użytkowania runtime (s) | Prymityw (tryb) |
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |------------- |
| Bernstein–Vazirani  |  50Q    | 100%  | Wskaźnik sukcesu (odsetek uruchomień, w których właściwa odpowiedź ma największą liczbę zliczeń)     | 10    | 8         | Sampler |
| Kwantowa transformata Fouriera | 30Q              | 100% | Wskaźnik sukcesu (odsetek uruchomień, w których właściwa odpowiedź ma największą liczbę zliczeń)      | 10    | 8        | Sampler |
| Kwantowe szacowanie fazy  | 30Q   | 99,9998%  | Dokładność znalezionego kąta: `1- abs(real_angle - angle_found)/pi`  | 10  | 8  | Sampler |
| Symulacja kwantowa: model Isinga (15 kroków) | 20Q  | 99,775%   |  $A$ (zdefiniowane poniżej)  |  60 (na krok)  | 15 (na krok)   | Estimator |
| Symulacja kwantowa 2: dynamika molekularna (20 punktów czasowych) | 34Q  |  96,78%  |  $A_{mean}$ (zdefiniowane poniżej)   | 10 (na punkt czasowy)    | 6 (na punkt czasowy)  | Estimator |

Definiując dokładność pomiaru wartości oczekiwanej – metryka $A$ jest zdefiniowana następująco:
$$
A = 1 - \frac{|\epsilon^{ideal} - \epsilon^{meas}|}{\epsilon^{ideal}_{max} - \epsilon^{ideal}_{min}},
$$
gdzie $ \epsilon^{ideal} $ = idealna wartość oczekiwana,  $ \epsilon^{meas} $ = zmierzona wartość oczekiwana, $\epsilon^{ideal}_{max} $ = idealna wartość maksymalna, a $\epsilon^{ideal}_{min}$ = idealna wartość minimalna. $A_{mean}$ jest po prostu średnią wartości $A$ z wielu pomiarów.

Metryka ta jest używana, ponieważ jest niezmiennicza względem globalnych przesunięć i skalowania zakresu osiągalnych wartości. Innymi słowy, niezależnie od tego, czy przesuniesz zakres możliwych wartości oczekiwanych w górę lub w dół, lub zwiększysz rozpiętość, wartość $A$ powinna pozostać spójna.
## Pierwsze kroki
Fire Opal Performance Management używa Qiskit w wersji v`2.0.0`, która jest wersją zalecaną. Obsługiwane wersje to Qiskit >=v`2.0.0`.
Uwierzytelnij się przy użyciu swojego [klucza API IBM Quantum Platform](http://quantum.cloud.ibm.com/) i wybierz funkcję Qiskit w następujący sposób. (Ten fragment kodu zakłada, że masz już [zapisane konto](/guides/functions#install-qiskit-functions-catalog-client) w lokalnym środowisku.)

In [1]:
# This cell is hidden from users. It hides the "...You have imported samplomatic..." warning.
import warnings

warnings.filterwarnings("ignore", message=".*You have imported samplomatic*")

In [2]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [3]:
# Access Function
perf_mgmt = catalog.load("q-ctrl/performance-management")

<Admonition type="note" title="Does this function support all IBM backends?">
If you want to use a backend that this function does not currently support, [reach out to Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) to add support.
</Admonition>

## Estimator primitive

### Estimator example

Use Fire Opal Performance Management's Estimator primitive to determine the expectation value of a single circuit-observable pair.

In addition to the `qiskit-ibm-catalog` and `qiskit` packages, you will also use the `numpy` package to run this example. You can install this package by uncommenting the following cell if you are running this example in a notebook using the IPython kernel.

In [4]:
# %pip install numpy

**2. Uruchom obwód**

Uruchom obwód i opcjonalnie zdefiniuj Backend oraz liczbę shotów.

In [5]:
import numpy as np
from qiskit.circuit.library import iqp
from qiskit.quantum_info import random_hermitian, SparsePauliOp

n_qubits = 50

# Generate a random circuit
mat = np.real(random_hermitian(n_qubits, seed=1234))
circuit = iqp(mat)
circuit.measure_all()

# Define observables as a string
observable = SparsePauliOp("Z" * n_qubits)

In [6]:
# Create PUB tuple
estimator_pubs = [(circuit, observable)]

Możesz użyć znanych [interfejsów API Qiskit Serverless](/guides/serverless), aby sprawdzić status zadania funkcji Qiskit:

In [7]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy().name

In [8]:
# Run the circuit using Estimator
qctrl_estimator_job = perf_mgmt.run(
    primitive="estimator",
    pubs=estimator_pubs,
    backend_name=backend_name,
)

**3. Pobierz wynik**

In [9]:
qctrl_estimator_job.status()

'QUEUED'

Wyniki mają taki sam format jak [wynik Estimator](/guides/estimator-input-output):

In [10]:
# Retrieve the counts from the result list
result = qctrl_estimator_job.result()

The results have the same format as an [Estimator result](/docs/guides/estimator-input-output):

In [22]:
import numpy

result_str = str(result)

with numpy.printoptions(threshold=200):
    print(
        f"The result of the submitted job had {len(result)} PUB "
        f"and has a value:\n {result[0]}\n"
    )

print("The associated PubResult of this job has the following DataBins:")
print(f"{result[0].data}\n")

print(f"And this DataBin has attributes: {result[0].data.keys()}")

print("The expectation values measured from this PUB are:")
print(f"{result[0].data.evs}")

The result of the submitted job had 1 PUB
The result of the submitted job had 1 PUB and has a value:
 PubResult(data=DataBin(evs=0.0195, stds=0.9998098569228051), metadata={'precision': None})

The associated PubResult of this job has the following DataBins:
DataBin(evs=0.0195, stds=0.9998098569228051)

And this DataBin has attributes: dict_keys(['evs', 'stds'])
The expectation values measured from this PUB are:
0.0195


## Prymityw Sampler
### Przykład użycia Samplera
Użyj prymitywu Sampler z Fire Opal Performance Management, aby uruchomić obwód Bernsteina–Vazirania. Algorytm ten, służący do znalezienia ukrytego ciągu znaków na podstawie wyników czarnej skrzynki, jest popularnym algorytmem do testów porównawczych, ponieważ istnieje tylko jedna poprawna odpowiedź.

**1. Utwórz obwód**

Zdefiniuj poprawną odpowiedź algorytmu — ukryty ciąg bitów — oraz obwód Bernsteina–Vazirania. Szerokość obwodu możesz łatwo zmieniać, modyfikując wartość `circuit_width`.

In [12]:
import qiskit

circuit_width = 35
hidden_bitstring = "1" * circuit_width

# Create circuit, reserving one qubit for BV oracle
bv_circuit = qiskit.QuantumCircuit(circuit_width + 1, circuit_width)
bv_circuit.x(circuit_width)
bv_circuit.h(range(circuit_width + 1))
for input_qubit, bit in enumerate(reversed(hidden_bitstring)):
    if bit == "1":
        bv_circuit.cx(input_qubit, circuit_width)
bv_circuit.barrier()
bv_circuit.h(range(circuit_width + 1))
bv_circuit.barrier()
for input_qubit in range(circuit_width):
    bv_circuit.measure(input_qubit, input_qubit)

# Create PUB tuple
sampler_pubs = [(bv_circuit,)]

**2. Uruchom obwód**

Uruchom obwód i opcjonalnie zdefiniuj Backend oraz liczbę strzałów (shots).

In [13]:
# Run the circuit using Sampler
qctrl_sampler_job = perf_mgmt.run(
    primitive="sampler",
    pubs=sampler_pubs,
    backend_name=backend_name,
)

Sprawdź [status](/guides/functions#check-job-status) swojego zadania Qiskit Function lub pobierz [wyniki](/guides/functions#retrieve-results) w następujący sposób:

In [14]:
# Print the ID so you can use it later, if necessary
print(qctrl_sampler_job.job_id)

qctrl_sampler_job.status()

60fe2fa1-a860-43e4-8615-c6ac4180f93b


'QUEUED'

**3. Retrieve the result**

In [15]:
# Retrieve the job results
sampler_result = qctrl_sampler_job.result()

In [16]:
# Get results for the first (and only) PUB
pub_result = sampler_result[0]
counts = pub_result.data.c.get_counts()

print("Counts for the meas output register (limited to 30 results):")
for i, (bitstring, count) in enumerate(counts.items()):
    if i >= 50:
        print(f"  ... ({len(counts) - 30} more items)")
        break
    print(f"  {bitstring}: {count}")

Counts for the meas output register (limited to 30 results):
  11111111111111111111111111111111111: 1661
  11111111111111111111111111110111111: 60
  11111111111111111111111111111101111: 54
  11111111111111111111111111111110111: 54
  11111111111111011111111111111111111: 46
  11111111111111111110111111111111111: 44
  11111111111111111111111101111111111: 42
  11111111111111111111111110111111111: 42
  11111111111111110111111111111111111: 41
  11111111111111111111111111111111101: 39
  11111111111111111111101111111111111: 38
  11111111111111111111110111111111111: 38
  11111111111111111111111111101111111: 37
  11111111111111111111111111111111110: 36
  11111111111110111111111111111111111: 35
  11111111111111111111111111111011111: 32
  11111111111111101111111111111111111: 32
  01111111111111111111111111111111111: 27
  11111111111111111011111111111111111: 23
  11111111101111111111111111111111111: 22
  11111111111111111111111111111111011: 21
  11111111011111111111111111111111111: 20
  00000000000

**3. Plot the top bitstrings**

Plot the bitstring with the highest counts to see if the hidden bitstring was the mode.

In [17]:
import matplotlib.pyplot as plt


def plot_top_bitstrings(counts_dict, hidden_bitstring=None):
    # Sort and take the top 100 bitstrings
    top_100 = sorted(counts_dict.items(), key=lambda x: x[1], reverse=True)[
        :100
    ]
    if not top_100:
        print("No bitstrings found in the input dictionary.")
        return

    # Unzip the bitstrings and their counts
    bitstrings, counts = zip(*top_100)

    # Assign colors: purple if the bitstring matches hidden_bitstring,
    # otherwise gray
    colors = [
        "#680CE9" if bit == hidden_bitstring else "gray" for bit in bitstrings
    ]

    # Create the bar plot
    plt.figure(figsize=(15, 8))
    plt.bar(
        range(len(bitstrings)), counts, tick_label=bitstrings, color=colors
    )

    # Rotate the bitstrings for better readability
    plt.xticks(rotation=90, fontsize=8)
    plt.xlabel("Bitstrings")
    plt.ylabel("Counts")
    plt.title("Top 100 Bitstrings by Counts")

    # Show the plot
    plt.tight_layout()
    plt.show()

The hidden bitstring is highlighted in purple, and it should be the bitstring with the highest number of counts.

In [18]:
plot_top_bitstrings(counts, hidden_bitstring)

<Image src="../docs/images/guides/q-ctrl-performance-management/extracted-outputs/8106d906-0.avif" alt="Output of the previous code cell" />

**3. Wykreśl najczęstsze ciągi bitów**

Wykreśl ciąg bitów o najwyższej liczbie wystąpień, aby sprawdzić, czy ukryty ciąg bitów był dominantą.